## ⚙️ Setup Instructions

Before running this notebook:

### 1. Complete Notebook 01 first
This notebook requires the merged dataset from Notebook 01.

### 2. Configure paths in `config/config.yaml`
- Input: `data.merged` (from Notebook 01)
- Output: `data.cleaned` (for Notebook 03)

### 3. Adjust settings (optional)
- `features.correlation_threshold`: Change filtering threshold (default: 0.95)
- `features.categorical`: Add/remove categorical features

### 4. Run cells in order!

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import yaml

## Configuration

In [ ]:
# Load configuration
config_path = Path("../config/config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

# Paths from config
PATH_INPUT = config['data']['merged']
PATH_OUTPUT = config['data']['cleaned']
PATH_FEATURES = config['data']['feature_list']

# Model configuration
TARGET = config['model']['target']
CORRELATION_THRESHOLD = config['features']['correlation_threshold']

# Categorical features (for proper handling)
CATEGORICAL_FEATURES = config['features']['categorical']

# RBP prefixes (protected from correlation removal)
RBP_PREFIXES = config['features']['rbp_prefixes']

print("✓ Configuration loaded from:", config_path)
print(f"\nInput:  {PATH_INPUT}")
print(f"Output: {PATH_OUTPUT}")
print(f"Target: {TARGET}")
print(f"Correlation threshold: {CORRELATION_THRESHOLD}")

## 1. Load Data

In [ ]:
print("="*80)
print("FEATURE CLEANING AND SELECTION")
print("="*80)

print("\nLoading data...")
df = pd.read_csv(PATH_INPUT)
print(f"✓ Loaded: {df.shape}")

# Separate target and features
y_full = df[TARGET].map({"TRUE": 1, "FALSE": 0, True: 1, False: 0}).astype("float")
mask = y_full.notna()
y = y_full.loc[mask].astype(int).reset_index(drop=True)
X = df.loc[mask].drop(columns=[TARGET]).reset_index(drop=True)

print(f"Valid samples: {len(y)}")
print(f"  Escapees: {y.sum()} ({y.sum()/len(y)*100:.1f}%)")
print(f"  NMD: {(~y.astype(bool)).sum()} ({(~y.astype(bool)).sum()/len(y)*100:.1f}%)")
print(f"Features: {X.shape[1]}")

## 2. Apply Custom Removal Rules

These rules reflect domain knowledge about NMD biology:
1. **AU vs GC content**: Keep AU (more biologically relevant), drop GC
2. **RBP features**: Keep ALL even if correlated (biologically meaningful)
3. **PTC distance**: Keep PTC.2.EJC (preferred), drop PTC_dist_exon_end_0b

In [ ]:
print("\n" + "="*80)
print("CUSTOM REMOVAL RULES")
print("="*80)

# Rename typo in feature name
if 'loefu_cat' in X.columns:
    X = X.rename(columns={'loefu_cat': 'loeuf_cat'})
    print("\n✓ Renamed 'loefu_cat' → 'loeuf_cat'")


# Manual drops (identifiers, leaky, redundant)
drop_manual = {
    "gene_length": "Identifier",
    "noncoding.length": "Identifier",
    "cds_length.x": "Duplicate",
    "cds_length.y": "Duplicate",
    "GENE_ID": "Identifier",
    "key": "Identifier",
    "txnames": "Identifier",
    "cds.nucloc": "Categorical indicator",
    "penultimate.exon": "Leaky - redundant with last.EJC",
    "last.exon": "Leaky - redundant with last.EJC",
    "downstream": "Redundant - less granular than last.EJC",
    "ALLELE.RAT": "Leaky - used in target calculation",
    "mRNAHalfLifeMin": "Redundant - half_life_PC1 kept",
    "mean_half_life": "Redundant - half_life_PC1 kept",
    "median_half_life": "Redundant - half_life_PC1 kept",
    "PTC.2.start.binning": "Redundant - unbinned version kept",
    "PTC.2.end.binning": "Redundant - unbinned version kept",
    "PTC.2.EJC.binning": "Redundant - unbinned version kept",
    "length.mutated.exon.binning": "Redundant - unbinned version kept",
    "EJC.downstream.cut": "Redundant - binned version not needed",
    "REVEL": "Not applicable - missense score for stopgain variants",
    # 5'UTR region-specific features (keep overall only)
    "FiveUTR_AUcontentlast200": "Redundant - overall fiveUTR_AU_content kept",
    "FiveUTR_AUcontentfirst200": "Redundant - overall fiveUTR_AU_content kept",
    "FiveUTR_UCcontentlast200": "Redundant - overall fiveUTR_UC_content kept",
    "FiveUTR_UCcontentfirst200": "Redundant - overall fiveUTR_UC_content kept",
    "fiveUTR_AUcontentlast100": "Redundant - overall fiveUTR_AU_content kept",
    "fiveUTR_AUcontentfirst100": "Redundant - overall fiveUTR_AU_content kept",
    "fiveUTR_UCcontentlast100": "Redundant - overall fiveUTR_UC_content kept",
    "fiveUTR_UCcontentfirst100": "Redundant - overall fiveUTR_UC_content kept",
    
    "aug_distance_nt": "Redundant - aug_distance_category captures this information",
    "COHORT_AF": "Data leakage - computed from training data, not generalizable"
}

print("\nRule 1: AU/GC content pairs")
print("  → Keep AU content (more biologically relevant)")
print("  → Drop GC content (redundant)")

gc_features_to_drop = [col for col in X.columns if '_GC_content' in col or 'GCcontent' in col]
print(f"  Dropping {len(gc_features_to_drop)} GC content features")

for col in gc_features_to_drop:
    drop_manual[col] = "Redundant - AU content kept (more biologically relevant)"

print("\nRule 2: RBP features")
print("  → Keep ALL RBP features (biologically meaningful correlation)")
print("  → Will be protected from automated correlation removal")

# RBP prefixes from config (protected from correlation removal)
rbp_prefixes = RBP_PREFIXES

print(f"RBP feature prefixes (protected): {len(rbp_prefixes)}")
for prefix in rbp_prefixes:
    print(f"  - {prefix}")
    
rbp_features = [col for col in X.columns if any(col.startswith(prefix) for prefix in rbp_prefixes)]
print(f"  Protecting {len(rbp_features)} RBP features")

print("\nRule 3: PTC distance features")
print("  → Keep PTC.2.EJC (preferred measure)")
print("  → Drop PTC_dist_exon_end_0b (less informative)")

if 'PTC_dist_exon_end_0b' in X.columns:
    drop_manual['PTC_dist_exon_end_0b'] = "Redundant - PTC.2.EJC kept (preferred)"
    print("  ✓ Will drop PTC_dist_exon_end_0b")
else:
    print("  ⚠️  PTC_dist_exon_end_0b not found")

# Apply manual drops
features_to_drop = [col for col in drop_manual.keys() if col in X.columns]
print(f"\nTotal manual drops: {len(features_to_drop)} features")

X_cleaned = X.drop(columns=features_to_drop)
print(f"After manual drops: {X_cleaned.shape}")

## 3. Automated Quality Checks

In [ ]:
print("\n" + "="*80)
print("AUTOMATED QUALITY CHECKS")
print("="*80)

automated_drops = []

# ============================================================================
# Check 1: Duplicate column names
# ============================================================================
print("\nCheck 1: Duplicate column names")
dup_cols = X_cleaned.columns[X_cleaned.columns.duplicated()].tolist()
if dup_cols:
    print(f"  ⚠️  Found {len(dup_cols)} duplicates:")
    for col in dup_cols:
        print(f"    - {col}")
    automated_drops.extend(dup_cols)
else:
    print("  ✓ None found")

# ============================================================================
# Check 2: Zero variance numeric features
# ============================================================================
print("\nCheck 2: Zero/near-zero variance numeric features")
zero_var = []
for col in X_cleaned.select_dtypes(include=[np.number]).columns:
    std = X_cleaned[col].std()
    if std < 1e-10:
        zero_var.append(col)

if len(zero_var) > 0:
    print(f"  Found {len(zero_var)} zero-variance features")
    # Show first 10
    for col in zero_var[:10]:
        print(f"    - {col}: constant = {X_cleaned[col].iloc[0]}")
    if len(zero_var) > 10:
        print(f"    ... and {len(zero_var)-10} more")
else:
    print("  ✓ None found")

automated_drops.extend(zero_var)

# ============================================================================
# Check 3: Constant categorical features
# ============================================================================
print("\nCheck 3: Constant categorical features")
print("  ⏭️  Skipped - will check after identifying categoricals in next step")

# ============================================================================
# Check 4: High correlation (WITH RBP PROTECTION)
# ============================================================================
print("\nCheck 4: High correlation (|r| > CORRELATION_THRESHOLD)")
print("  Note: RBP features are PROTECTED from removal")

numeric_cols = X_cleaned.select_dtypes(include=[np.number]).columns
high_corr_drops = []
rbp_pairs_protected = 0

if len(numeric_cols) > 1:
    print("  Computing correlation matrix...")
    corr_matrix = X_cleaned[numeric_cols].corr().abs()
    
    print("  Checking correlations...")
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if corr_matrix.iloc[i, j] > CORRELATION_THRESHOLD:
                feat1 = corr_matrix.columns[i]
                feat2 = corr_matrix.columns[j]
                
                # Check if RBP features
                is_rbp1 = any(feat1.startswith(prefix) for prefix in rbp_prefixes)
                is_rbp2 = any(feat2.startswith(prefix) for prefix in rbp_prefixes)
                
                if is_rbp1 and is_rbp2:
                    # Both RBP - keep both
                    rbp_pairs_protected += 1
                elif is_rbp1:
                    # Protect feat1, drop feat2
                    if feat2 not in high_corr_drops:
                        high_corr_drops.append(feat2)
                        print(f"    Drop {feat2}, keep {feat1} (RBP protected)")
                elif is_rbp2:
                    # Protect feat2, drop feat1
                    if feat1 not in high_corr_drops:
                        high_corr_drops.append(feat1)
                        print(f"    Drop {feat1}, keep {feat2} (RBP protected)")
                else:
                    # Neither RBP - drop second
                    if feat2 not in high_corr_drops:
                        high_corr_drops.append(feat2)
                        print(f"    Drop {feat2}, correlated with {feat1} (r={corr_matrix.iloc[i, j]:.3f})")
    
    print(f"\n  ✓ Protected {rbp_pairs_protected} RBP-RBP pairs")
    print(f"  ✓ Dropping {len(high_corr_drops)} correlated features")
else:
    print("  Not enough numeric features to check")

automated_drops.extend(high_corr_drops)

# Remove duplicates
automated_drops = list(set(automated_drops))
print(f"\nTotal automated drops: {len(automated_drops)}")

## 4. Prepare Categorical Features

In [ ]:
print("\n" + "="*80)
print("CATEGORICAL FEATURE PREPARATION")
print("="*80)

# Categorical features from config
declared_cat = CATEGORICAL_FEATURES

print(f"Categorical features from config: {len(declared_cat)}")
for feat in declared_cat:
    print(f"  - {feat}")

# Find binary numeric features (should be treated as categorical)
print("\nDetecting binary features...")
binary_features = []
for col in X_cleaned.select_dtypes(include=[np.number]).columns:
    unique_vals = X_cleaned[col].dropna().unique()
    if len(unique_vals) == 2:
        # Check if it's 0/1, True/False, or similar binary
        if set(unique_vals).issubset({0, 1, 0.0, 1.0, True, False}):
            binary_features.append(col)

if len(binary_features) > 0:
    print(f"\nFound {len(binary_features)} binary features (will treat as categorical):")
    for col in sorted(binary_features)[:10]:  # Show first 10
        vals = X_cleaned[col].value_counts().sort_index().to_dict()
        print(f"  {col}: {vals}")
    if len(binary_features) > 10:
        print(f"  ... and {len(binary_features) - 10} more")

# Build complete categorical feature list
cat_features = [c for c in declared_cat if c in X_cleaned.columns]
cat_features.extend(binary_features)

# Find other object-type columns
other_objs = [c for c in X_cleaned.columns 
              if X_cleaned[c].dtype == "object" 
              and c not in cat_features]
cat_features.extend(other_objs)

# Remove duplicates
cat_features = list(set(cat_features))

print(f"\nTotal categorical features: {len(cat_features)}")
print(f"  Declared: {len([c for c in declared_cat if c in X_cleaned.columns])}")
print(f"  Binary: {len(binary_features)}")
print(f"  Other objects: {len(other_objs)}")

# Convert ALL categoricals to string and handle missing
print("\nConverting categoricals to strings and handling missing values...")
for col in cat_features:
    X_cleaned[col] = X_cleaned[col].astype(str)
    X_cleaned[col] = X_cleaned[col].replace('nan', 'MISSING')
    if X_cleaned[col].isna().any():
        X_cleaned[col] = X_cleaned[col].fillna('MISSING')

# Save missingness info BEFORE imputation (for feature list later)
print("\nRecording missingness before imputation...")
feature_missingness = {}
for col in X_cleaned.columns:
    non_null = X_cleaned[col].notna().sum()
    feature_missingness[col] = {
        'non_null_count': non_null,
        'non_null_pct': non_null / len(X_cleaned) * 100
    }


# Impute remaining numeric features with median
numeric_cols = X_cleaned.select_dtypes(include=[np.number]).columns
missing_numeric = X_cleaned[numeric_cols].isna().sum()
missing_numeric = missing_numeric[missing_numeric > 0]

if len(missing_numeric) > 0:
    print(f"\nImputing {len(missing_numeric)} numeric features with median...")
    for col in missing_numeric.index:
        median_val = X_cleaned[col].median()
        X_cleaned[col] = X_cleaned[col].fillna(median_val)
        print(f"  {col}: {missing_numeric[col]} → median={median_val:.4f}")

print(f"\n✓ Remaining NaN: {X_cleaned.isna().sum().sum()}")
print(f"✓ Final feature breakdown:")
print(f"  Categorical: {len(cat_features)}")
print(f"  Numeric: {len(X_cleaned.select_dtypes(include=[np.number]).columns)}")

## 5. Apply Final Drops and Create Final Dataset

In [ ]:
print("\n" + "="*80)
print("CREATING FINAL DATASET")
print("="*80)

# Apply automated drops
X_final = X_cleaned.drop(columns=[col for col in automated_drops if col in X_cleaned.columns])

print(f"\nFinal shape: {X_final.shape}")
print(f"  Samples: {len(X_final)}")
print(f"  Features: {X_final.shape[1]}")

## 6. Cleaning Summary

In [ ]:
print("\n" + "="*80)
print("CLEANING SUMMARY")
print("="*80)

print(f"\nFeature counts:")
print(f"  Original: {X.shape[1]}")
print(f"  After manual drops: {X_cleaned.shape[1]}")
print(f"  After automated drops: {X_final.shape[1]}")
print(f"\n  Total removed: {X.shape[1] - X_final.shape[1]}")
print(f"    - Manual: {len(features_to_drop)}")
print(f"    - Automated: {len(automated_drops)}")

# Verify custom rules
print("\n" + "="*80)
print("CUSTOM RULE VERIFICATION")
print("="*80)

# Check RBP features retained
rbp_retained = [col for col in X_final.columns if any(col.startswith(prefix) for prefix in rbp_prefixes)]
print(f"\n✓ RBP features retained: {len(rbp_retained)}")
print(f"  (Original: {len(rbp_features)})")

# Check AU content retained
au_retained = [col for col in X_final.columns if '_AU_content' in col or 'AUcontent' in col]
print(f"\n✓ AU content features retained: {len(au_retained)}")

# Check GC content dropped
gc_remaining = [col for col in X_final.columns if '_GC_content' in col or 'GCcontent' in col]
if len(gc_remaining) == 0:
    print(f"✓ GC content features dropped: {len(gc_features_to_drop)}")
else:
    print(f"⚠️  GC content features remaining: {len(gc_remaining)}")
    print(f"  Should be 0!")

# Check PTC.2.EJC retained
if 'PTC.2.EJC' in X_final.columns:
    print(f"\n✓ PTC.2.EJC retained (preferred measure)")
else:
    print(f"\n⚠️  PTC.2.EJC missing!")

if 'PTC_dist_exon_end_0b' in X_final.columns:
    print(f"⚠️  PTC_dist_exon_end_0b still present (should be dropped)")
else:
    print(f"✓ PTC_dist_exon_end_0b dropped")

# Check codon optimality features present
print("\n" + "="*80)
print("NEW FEATURE VERIFICATION")
print("="*80)

codon_features = ['CodonOptimalityFraction_CDS', 'CodonOptimalityFraction_PTCpm100nt']
for feat in codon_features:
    if feat in X_final.columns:
        non_null = X_final[feat].notna().sum()
        print(f"\n✓ {feat}")
        print(f"  Non-null: {non_null}/{len(X_final)} ({non_null/len(X_final)*100:.1f}%)")
        print(f"  Mean: {X_final[feat].mean():.4f}")
        print(f"  Range: [{X_final[feat].min():.4f}, {X_final[feat].max():.4f}]")
    else:
        print(f"\n⚠️  {feat} MISSING!")

# Check AverageCodonRNAUsage is gone
if 'AverageCodonRNAUsage' in X_final.columns:
    print(f"\n⚠️  AverageCodonRNAUsage still present (should be dropped!)")
else:
    print(f"\n✓ AverageCodonRNAUsage dropped (replaced by codon optimality)")

## 7. Save Cleaned Dataset and Feature List

In [ ]:
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

# Combine features and target
final_df = X_final.copy()
final_df[TARGET] = y.values

# Save cleaned dataset
final_df.to_csv(PATH_OUTPUT, index=False)
print(f"\n✓ Cleaned dataset saved: {PATH_OUTPUT}")
print(f"  Rows: {len(final_df)}")
print(f"  Columns: {final_df.shape[1]}")

# Create and save feature list
feature_list = pd.DataFrame({
    'feature': X_final.columns,
    'dtype': [str(X_final[col].dtype) for col in X_final.columns],
    'is_categorical': [col in cat_features for col in X_final.columns],
    'non_null_count': [feature_missingness.get(col, {}).get('non_null_count', len(X_final)) for col in X_final.columns],
    'non_null_pct': [feature_missingness.get(col, {}).get('non_null_pct', 100.0) for col in X_final.columns]
})

# Add summary stats for numeric features
feature_list['mean'] = np.nan
feature_list['std'] = np.nan
feature_list['min'] = np.nan
feature_list['max'] = np.nan

for idx, row in feature_list.iterrows():
    if not row['is_categorical'] and row['dtype'] in ['float64', 'int64']:
        col = row['feature']
        feature_list.at[idx, 'mean'] = X_final[col].mean()
        feature_list.at[idx, 'std'] = X_final[col].std()
        feature_list.at[idx, 'min'] = X_final[col].min()
        feature_list.at[idx, 'max'] = X_final[col].max()

feature_list.to_csv(PATH_FEATURES, index=False)
print(f"\n✓ Feature list saved: {PATH_FEATURES}")

print("\n" + "="*80)
print("COMPLETE! 🎉")
print("="*80)

print(f"""
Dataset ready for model training:
  - Samples: {len(final_df)}
  - Features: {X_final.shape[1]}
  - Target: {TARGET}

Custom rules applied:
  ✓ Kept AU content, dropped GC content
  ✓ Protected ALL RBP features (biologically meaningful)
  ✓ Kept PTC.2.EJC over PTC_dist_exon_end_0b
  ✓ Replaced AverageCodonRNAUsage with codon optimality features

Ready for model training! (notebook 03)
""")